# Building Watchtower2: Architecture and Development Sequence


Watchtower2 begins with a concrete problem: ordinary CMS work can add uploaded files, task history and recoverable copies to Git, while each build can allocate another large generated project on local disk. A system can behave correctly for one save and still become difficult to maintain after thousands of saves.

The course develops an independent CMS prototype around that problem. It also addresses a second failure: accepting AI-generated code without understanding which component owns a decision or what happens when a process stops halfway through it. Each implementation stage must leave behind both working behavior and an explanation its owner can inspect.

::: {.callout-note title="A proposed system, developed in small stages"}
The current Watchtower CMS works through a CLI and local web interface. Watchtower2 currently has a project scaffold and design documents. The architecture below is the target to investigate, not a description of completed v2 code. This overview is Draft; the later chapters are Planned. Prototype adoption or migration is a separate future decision.
:::


## The application and the learning contract

A **content-management system** coordinates more than page text. Watchtower holds artifact identity, titles, public summaries, visibility, publication state, course organization, portfolio entries, profile information, photos, author notes, to-dos and linked Kanban tasks. It also manages uploaded files and turns notebooks into a website.

These features create useful constraints. A file may belong to both a task and an article. A draft chapter may be hidden because its course is not published. A notebook can be saved by an editor while an operation is running. A failed build must leave the previous site usable. These are observable product behaviors with consequences for the architecture.

The implementation lives in `projects/watchtower2/`. The working v1 CMS supplies examples to study through read-only traces and copied fixtures. Prototype mutations use separate databases, object namespaces, source copies, credentials and output targets. Sharing a repository does not make the two systems share operational state.

The course treats **AI-assisted development** as a way of doing the engineering. Watchtower2 does not need a model in its application request path. An assistant can locate code, propose a schema, implement a bounded change or suggest a failure case. The owner must still be able to explain the authority, the state transition and the evidence before accepting a consequential decision. No particular assistant or agent topology is part of the application design.


## Storage authority starts with the kind of information

Putting all state in YAML makes changes visible in Git, but frequent operational updates and large payloads then become part of repository history. Deleting a file in a later commit does not remove its earlier bytes. Conversely, ignoring every CMS file would remove the history without providing a database, ownership model or recovery contract.

Watchtower2 assigns each kind of information an explicit authority:

| Authority | Information it owns | Why it belongs there |
| --- | --- | --- |
| Git | Application code, schema changes, authored content notebooks and sidecars, explicit public release manifests | These are reviewed source changes or deliberate publication snapshots. |
| Turso | Artifact metadata, lifecycle, plans, notes, ordered relationships, course outlines, photos, tasks, object ownership, pending operations and retention records | These are mutable application records that need constraints, scoped versions and transactions. |
| Backblaze B2 | Managed attachment/image bytes and explicitly retained backup/export payloads | Payload storage is independent from the number and lifetime of records that reference it. |
| Ignored local storage | Generated frontend notebooks, hydrated assets, rendered stages, caches and operation staging | These are disposable or recoverable working files with explicit leases and cleanup rules. |

An active task or private note is durable user data even if it changes often. Moving it to a database does not authorize age-based deletion. A release asset may remain necessary after it disappears from the current authoring view. Retention follows ownership and promises, rather than a guess that something looks temporary.

Ordinary card, note, photo and metadata saves should create no tracked file changes in v2. Editing an authored notebook or preparing a release can create a Git diff. Source notebook outputs can still be large, and growing authored content remains legitimate growth. This design limits new operational accumulation; it does not shrink existing Git history automatically.


## Two interfaces over one application core

The **application core** implements use cases such as move a card, attach a file, start an artifact or retire a release. It applies domain rules and coordinates adapters for database access, object storage, source files and rendering. The CLI and Flet are two ways to submit the same intent.

```{mermaid}
flowchart TB
    CLI["wt2 CLI"] --> CORE["Shared application use cases"]
    UI["Local declarative Flet CMS"] --> CORE
    WORK["Active-session worker and explicit cleanup"] --> CORE
    CORE --> RULES["Domain rules and valid state transitions"]
    CORE --> DB["Turso records and operation state"]
    CORE --> BLOB["B2 verified object versions"]
    CORE --> SOURCE["Bound source workspace: content notebooks and code"]
    CORE --> BUILD["Projection and rendering adapters"]
```

The arrows describe calls and coordination, not one transaction across every destination. Domain rules do not import Flet or a storage SDK. A Flet event handler does not independently decide which B2 key can be deleted. The backend owns deletion checks so a direct CLI action and a policy worker cannot disagree about safety. The CLI remains usable without installing or starting Flet; a permanently running web server is not required for every operation.

In a **declarative interface**, controls are derived from view state. The companion **Introduction to Flet** post series covers the general component, state, form, async and navigation mechanics; those posts are currently Planned. Here the concern is Watchtower2's boundary: editor buffers, the loaded revision, validation errors and a pending save belong in view state, while the application core owns accepted records. A component re-render does not establish that a save succeeded. The capability chapter will exercise the actual CMS interactions on a pinned runtime. [Flet's declarative model](https://flet.dev/docs/cookbook/declarative-vs-imperative/).

Turso is the proposed remote record store. Its Python documentation distinguishes local and remote access and drivers for different engines. The course therefore chooses and exercises a specific engine/driver combination before relying on transaction behavior. A local SQLite adapter is useful for fast disposable checks; it does not prove the remote contract. The initial design uses authoritative remote writes and leaves offline writable synchronization for a later conflict model. [Turso Python quickstart](https://docs.turso.tech/sdk/python/quickstart).


## A save has a decision boundary

Suppose a card editor reads board version 12. Another client moves a card and commits version 13. The original editor must not silently save its older decision over the new board. An **expected revision** identifies the state on which that decision was based; a conditional transaction accepts the mutation only when the relevant revision still matches.

A conflict preserves the submitted fields and explains that the saved state changed. The person can read the new state and decide how to proceed. Retrying the stale edit automatically would make that decision on their behalf. Versions are scoped: an unrelated profile edit should not invalidate a card-only operation.

A different problem occurs when the connection fails after a possible commit. An **operation ID** identifies a particular request, allowing the client to inspect its recorded result or replay that same request without repeating its effect. It does not replace the expected revision. The receipt is bound to the request payload, and an expired request must remain invalid even after old receipts are pruned.

The first complete slice will make these distinctions visible with a small Kanban workflow. The learner will inspect the changed rows, predict which client wins and explain what the losing client sees. This gives the Flet interface a proven application contract before many screens depend on it.


## Files introduce intermediate states

A database row and a B2 upload cannot commit atomically. An **upload intent** records the attempted operation before bytes are sent. A physical upload has a unique key and version identity. Only after its length and content digest are verified can a short database transaction create a ready ownership link, rechecking the owner's revision.

An **owner link** says that a particular artifact, card or image field needs the bytes. It also holds that owner's filename. Two owners can share one verified payload, and promoting a task attachment to an entity adds a link rather than another copy. A logical digest identifies content; a physical key/version identifies the particular stored object that a worker may later remove. A distinct upload attempt uses a distinct physical identity even when its digest matches.

If metadata validation fails after upload, the system records an unattached attempt. It can be reconciled and collected after the appropriate grace period. It must not create an apparently ready owner whose bytes were never verified. These intermediate states are part of the user-visible result and recovery model.

Source notebooks introduce another boundary. Starting an artifact may require both a source file and database state. A durable intent, expected source absence or hash, ignored staging and conditional installation allow recovery to reason about both stores. Matching proposed bytes can complete an interrupted operation. A divergent editor save must be preserved for explicit resolution. The system must not seed a started notebook again or overwrite a conflicting save to make the record look consistent.


## Garbage collection protects promises before reclaiming bytes

**Garbage collection** finds storage that the application is allowed to remove. Its protecting references are called **roots**. Active owners, retained undo records, exported releases, unexpired backups and valid pending operations can all be roots. A zero live-owner count is only the beginning of the eligibility check.

Consider a file referenced by a card and an article. Removing the card's link leaves the article working. Removing the article's link can begin quarantine, but a retained release or backup may still need the exact version. The backend must be able to explain that protection. A timer expiring is insufficient authority to delete it.

```{mermaid}
flowchart LR
    U["Upload intent"] --> V["Verified version"]
    V --> O["Owned bytes"]
    O --> Q["Last live owner removed: quarantine"]
    Q --> O
    Q --> C["All roots absent: deletion claim"]
    C --> D["Exact version absent: recorded deletion"]
```

A deletion claim is a transactional state transition that prevents new ownership of that physical version. A competing attachment must win before the claim or use another verified version. This makes delayed workers and replacement uploads manageable. B2's S3 API distinguishes a version-specific permanent deletion from a delete without a version ID, which inserts a delete marker. The collector must account for versions and markers rather than equating a hidden key with reclaimed storage. [B2 deletion semantics](https://www.backblaze.com/apidocs/s3-delete-object).

Proposed starting policies include 24 hours for abandoned uploads, at least seven days of last-owner quarantine and 30 days of undo/backup retention. These are design defaults to review and exercise, not enabled promises. Release and backup protection can keep bytes longer. The backend exposes dry runs, blocking reasons, per-run limits and retryable failures; direct deletion uses the same eligibility rules as automatic collection.

Cleanup runs at startup, after relevant operations, periodically during active use and on explicit request. The operating assumption is no authoring activity while the laptop is closed. Eligibility can expire during inactivity and be handled by bounded catch-up next session. Unavailable or inconsistent root inventory stops destructive work.

The recovery chapter must demonstrate that the oldest supported backup still finds every required object version after live owners disappear. Until that promise is proven, destructive GC remains a disposable-resource experiment. Completed task status alone never proves that its results were handed off or its files can be purged.


## Content notebooks, frontend notebooks and the website

A **content notebook** is the authored source: body cells, saved outputs, inline attachments, execution metadata and cell-level options. A **frontend notebook** is a generated copy with CMS-derived document metadata, wrappers and projected links. The source notebook remains authoritative for its body; changes to generated copies never synchronize back into it.

```{mermaid}
flowchart TB
    SOURCE["Content notebooks: authored Git source"] --> SNAP["Captured build inputs"]
    LIVE["Preview: saved CMS metadata"] --> SNAP
    MANIFEST["Production: frozen public manifest"] --> SNAP
    ASSETS["Selected verified B2 site assets"] --> SNAP
    SNAP --> GENERATOR["Frontend projection generator"]
    GENERATOR --> NB["Frontend notebooks: generated and ignored"]
    GENERATOR --> QMD["Generated QMD homes, listings and navigation"]
    NB --> QUARTO["Quarto renders stored outputs; no kernel run"]
    QMD --> QUARTO
    QUARTO --> STAGE["Isolated rendered stage"]
    STAGE --> PROMOTE["Promote only after success"]
```

Preview captures eligible live authoring metadata and saved source files. Production uses committed sources plus a deliberately frozen public release manifest. Both feed the same projection/render boundary. Course homes and listings can be generated QMD pages; they need not become additional source notebooks. Private course parents and draft publication states affect eligibility, links and assets as well as page lists.

A generated stage can be large because frontend copies include notebook outputs and hydrated assets. It needs a **lease**, an explicit claim that active work still needs the stage. Cleanup protects running renderers and exports, retains the current site and a successful fallback, and bounds old stages, logs and caches. A failed render or asset download leaves the previous successful output available. Building must leave content notebook hashes unchanged.


## A release is an explicit snapshot

Ordinary authoring saves are separate from deployment. Preparing a **release manifest** selects allowlisted public metadata, eligible source paths and digests, exact site-asset versions and rendering configuration identity. Plans, internal notes, task cards, context attachments and credentials are excluded. Relevant object versions are pinned before the manifest is exported.

The source files and inspected manifest diff can then be committed for a deliberate release. Merging unrelated code into `main` need not create a new snapshot of whatever happens to be in the live database. The prototype must implement an explicit release trigger and demonstrate that distinction in an isolated publishing target. The manifest uses source digests; a later receipt can bind its digest to the resulting Git commit without trying to embed that containing commit's identity inside itself.

The proposed CI build reads Git and narrowly scoped read-only B2 site-input credentials. It receives no Turso token and no access to private context attachments. GitHub deployment permissions are separate from B2 read permissions. The workflow generates frontend notebooks, renders the site and uploads a Pages deployment artifact. GitHub documents artifact-based custom Pages workflows; this course will demonstrate one without changing the current website's workflow or domain. [GitHub Pages custom workflows](https://docs.github.com/en/pages/getting-started-with-github-pages/using-custom-workflows-with-github-pages).

Rebuilding every historical release forever would keep its required assets forever. The course instead makes the supported rebuild window an explicit product decision, with current/fallback releases and named pins protected until recorded retirement. An old manifest can remain in Git after its assets have legitimately expired; it must no longer carry an unqualified rebuild guarantee. Backup retention and release retention must agree before collection removes the last protected version.


## The sequence makes the risks observable

The ordered course outline is the navigation authority. Its development sequence follows dependencies rather than attempting to complete every layer before observing a working feature.

1. **Understand and design, Chapters 01–03.** Trace v1, measure the different kinds of growth, write the authority map and invariants, then run disposable Turso, B2 and Flet capability experiments. The outputs are verified observations and small architectural decisions, with unsupported assumptions left visible.
2. **Build one complete slice, Chapters 04–06.** Implement Kanban through a local headless core and `wt2`, establish the hosted transaction/retry contract, then connect a thin real Flet screen. Demonstrate both interfaces observing the same saved state and rejecting the same stale decision before expanding the feature set.
3. **Own files and recover safely, Chapters 07–09.** Add verified upload attempts and ownership, one backend deletion service, configurable policies, durable jobs and restore drills. The completion evidence is an explained race or interruption with inspected records and bytes. GC stays isolated while its safety guarantees are under development.
4. **Develop authoring workflows, Chapters 10–13.** Extend the same boundaries to catalog identity, plans and memory, notebook/source operations, ordered courses, portfolio/profile/photos and linked task context. Each sub-slice gets a focused CLI/core contract and its Flet surface. Feature parity is tracked as observed behavior, including intentional interface changes.
5. **Build and release the website, Chapters 14–15.** Generate disposable frontend notebooks and preview output, bound build storage, then freeze a release and build it without the live database. Show source bytes unchanged, internal data excluded and the last good output preserved during failure.
6. **Audit and evaluate, Chapter 16.** Deliberately violate selected rules in a scratch copy, establish which checks detect them, compare v1/v2 behavior and inspect storage over repeated cycles. Report limitations and choose future experiments from evidence. Adoption does not follow automatically.

This sequence is a teaching plan, not a time estimate. Provider behavior or a failed assumption may require a smaller experiment before the next feature. Record that change with its reason rather than presenting the original plan as an accomplished result.


## The review loop and the standard of evidence

For each small change, first read the relevant trace and current code. State the expected input, state transition, affected files/records and failure cases. Ask the assistant for a bounded implementation plan and review consequential choices before implementing. Then inspect the diff and actual stored effects, demonstrate the important rule, explain the result and record unresolved questions with a next action.

Different evidence supports different claims. A pure unit check establishes a narrow rule. A provider contract check exercises a selected remote configuration. A user-interface demonstration establishes observable interaction on the recorded runtime. A restore drill validates a particular retained snapshot and its files. None of these alone establishes the whole application.

| Claim | Evidence the course will require |
| --- | --- |
| Both interfaces obey one rule | Trace the same use case from CLI and Flet and compare stored effects and failures. |
| Stale edits do not overwrite decisions | Competing clients, one conditional winner and unchanged rows for the loser. |
| Uploaded bytes are owned safely | Verified version identity, inspected owner links and interrupted-finalization recovery. |
| Collection is safe and useful | Protected-root explanations, attach/delete races, exact-version absence and repeated-cycle accounting. |
| Publication is isolated | A frozen release builds without Turso, excludes internal fields and preserves the last good output on failure. |
| The owner understands the application | Predict a transition, trace its implementation and diagnose a demonstrated failure without accepting an agent summary as proof. |

Architecture checks must be seen rejecting an intentional violation. They still have limits: import checks cannot judge every abstraction, and a passing suite cannot decide how long old releases should remain rebuildable. Human review covers those decisions and checks whether the report's claims match its evidence. The course makes no prior claim of time saved or understanding gained.

The initial scope is a local authoring CMS with remote records and object storage, one explicitly bound writable checkout, and isolated website experiments. A public multi-user hosted CMS, offline writable synchronization and automatic v1 adoption require additional design. The first implementation chapter begins with the narrower task that makes all later choices credible: explain what the current system actually does.
